# Steam duplicate user rows EDA

This notebook inspects duplicate user records in the Steam raw file `australian_users_items.json`.

The file stores user library snapshots, not timestamped interaction events. For canonicalization, the main question is whether repeated rows are different users, exact duplicate snapshots, or multiple snapshots of the same Steam account.

In [1]:
from __future__ import annotations

import ast
import gzip
import hashlib
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "beyond_click_sim").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
RAW_DIR_CANDIDATES = [
    REPO_ROOT.parents[1] / "data" / "raw" / "steam",
    REPO_ROOT / "data" / "raw" / "steam",
]
RAW_DIR = next((path for path in RAW_DIR_CANDIDATES if path.exists()), None)
if RAW_DIR is None:
    raise FileNotFoundError(RAW_DIR_CANDIDATES)

RAW_USERS_PATH = RAW_DIR / "australian_users_items.json"
RAW_GAMES_PATH = RAW_DIR / "steam_games.json.gz"
assert RAW_USERS_PATH.exists(), RAW_USERS_PATH
assert RAW_GAMES_PATH.exists(), RAW_GAMES_PATH

REPO_ROOT, RAW_USERS_PATH, RAW_GAMES_PATH

(PosixPath('/Users/a.agafonov/Research/agent_simulator/repos/beyond-click-sim'),
 PosixPath('/Users/a.agafonov/Research/agent_simulator/data/raw/steam/australian_users_items.json'),
 PosixPath('/Users/a.agafonov/Research/agent_simulator/data/raw/steam/steam_games.json.gz'))

## Raw data examples

Steam raw data has two files:

- `australian_users_items.json`: one user library snapshot per line, with nested owned games;
- `steam_games.json.gz`: one game metadata record per line.

In [2]:
with RAW_USERS_PATH.open(encoding="utf-8", errors="ignore") as file:
    for user_example_line, line in enumerate(file, start=1):
        user_example = ast.literal_eval(line)
        if user_example.get("items"):
            break

user_record = {key: value for key, value in user_example.items() if key != "items"}
user_record["source_line"] = user_example_line
user_record["observed_items_count"] = len(user_example["items"])

display(pd.DataFrame([user_record]))
display(pd.DataFrame(user_example["items"][:5]))

,user_id,items_count,steam_id,user_url,source_line,observed_items_count
0,76561197970982479,277,76561197970982479,http://steamcommunity.com/profiles/76561197970...,1,277


,item_id,item_name,playtime_forever,playtime_2weeks
0,10,Counter-Strike,6,0
1,20,Team Fortress Classic,0,0
2,30,Day of Defeat,7,0
3,40,Deathmatch Classic,0,0
4,50,Half-Life: Opposing Force,0,0


The first table is the user-level record. The second table is the nested library list that becomes canonical `interactions`: one row per `(user, item)` ownership snapshot.

In [3]:
with gzip.open(RAW_GAMES_PATH, "rt", encoding="utf-8", errors="ignore") as file:
    for game_example_line, line in enumerate(file, start=1):
        game_example = ast.literal_eval(line)
        if game_example.get("id"):
            break

pd.DataFrame(
    {
        "field": key,
        "python_type": type(value).__name__,
        "value_preview": repr(value)[:180],
    }
    for key, value in game_example.items()
)

,field,python_type,value_preview
0,publisher,str,'Kotoshiro'
1,genres,list,"['Action', 'Casual', 'Indie', 'Simulation', 'S..."
2,app_name,str,'Lost Summoner Kitty'
3,title,str,'Lost Summoner Kitty'
4,url,str,'http://store.steampowered.com/app/761140/Lost...
5,release_date,str,'2018-01-04'
6,tags,list,"['Strategy', 'Action', 'Indie', 'Casual', 'Sim..."
7,discount_price,float,4.49
8,reviews_url,str,'http://steamcommunity.com/app/761140/reviews/...
9,specs,list,['Single-player']


## Helpers

The raw file is line-delimited Python literal syntax, so `ast.literal_eval` is safer here than `json.loads`.

In [4]:
def norm_id(value: object) -> str:
    return "" if value is None else str(value)


def as_int(value: object) -> int:
    return 0 if value in (None, "") else int(value)


def iter_user_rows(path: Path):
    with path.open(encoding="utf-8", errors="ignore") as file:
        for line_no, line in enumerate(file, start=1):
            yield line_no, ast.literal_eval(line)


def item_map(items: list[dict]) -> dict[str, tuple[int, int, str]]:
    return {
        norm_id(item.get("item_id")): (
            as_int(item.get("playtime_forever")),
            as_int(item.get("playtime_2weeks")),
            norm_id(item.get("item_name")),
        )
        for item in items
    }


def digest_items(items: list[dict], *, include_playtime: bool) -> str:
    digest = hashlib.sha1()
    for item in sorted(items, key=lambda item: norm_id(item.get("item_id"))):
        digest.update(norm_id(item.get("item_id")).encode())
        digest.update(b"\0")
        if include_playtime:
            digest.update(str(as_int(item.get("playtime_forever"))).encode())
            digest.update(b"\0")
            digest.update(str(as_int(item.get("playtime_2weeks"))).encode())
            digest.update(b"\0")
            digest.update(norm_id(item.get("item_name")).encode())
        digest.update(b"\n")
    return digest.hexdigest()


def row_summary(line_no: int, row: dict) -> dict:
    items = row.get("items") or []
    return {
        "source_line": line_no,
        "steam_id": norm_id(row.get("steam_id")),
        "raw_user_id": norm_id(row.get("user_id")),
        "user_url": norm_id(row.get("user_url")),
        "reported_items_count": as_int(row.get("items_count")),
        "observed_items_count": len(items),
        "total_playtime_forever": sum(as_int(item.get("playtime_forever")) for item in items),
        "total_playtime_2weeks": sum(as_int(item.get("playtime_2weeks")) for item in items),
        "item_id_digest": digest_items(items, include_playtime=False),
        "full_snapshot_digest": digest_items(items, include_playtime=True),
    }

## First pass: compact summary for all user rows

In [5]:
groups_by_steam_id: dict[str, list[dict]] = defaultdict(list)
groups_by_raw_user_id: dict[str, list[dict]] = defaultdict(list)

raw_user_rows = 0
raw_owned_rows = 0
raw_played_rows = 0
raw_zero_playtime_rows = 0

for line_no, row in iter_user_rows(RAW_USERS_PATH):
    summary = row_summary(line_no, row)
    raw_user_rows += 1
    raw_owned_rows += summary["observed_items_count"]
    groups_by_steam_id[summary["steam_id"]].append(summary)
    groups_by_raw_user_id[summary["raw_user_id"]].append(summary)

    for item in row.get("items") or []:
        if as_int(item.get("playtime_forever")) > 0:
            raw_played_rows += 1
        else:
            raw_zero_playtime_rows += 1

overview = pd.DataFrame(
    [
        ("raw_user_rows", raw_user_rows),
        ("unique_steam_id", len(groups_by_steam_id)),
        ("unique_raw_user_id", len(groups_by_raw_user_id)),
        ("raw_owned_rows", raw_owned_rows),
        ("raw_played_rows", raw_played_rows),
        ("raw_zero_playtime_rows", raw_zero_playtime_rows),
    ],
    columns=["metric", "value"],
)
overview

,metric,value
0,raw_user_rows,88310
1,unique_steam_id,87625
2,unique_raw_user_id,87626
3,raw_owned_rows,5153209
4,raw_played_rows,3285246
5,raw_zero_playtime_rows,1867963


## Duplicate counts by id

In [6]:
duplicate_steam_groups = {steam_id: rows for steam_id, rows in groups_by_steam_id.items() if len(rows) > 1}
duplicate_raw_user_groups = {user_id: rows for user_id, rows in groups_by_raw_user_id.items() if len(rows) > 1}

duplicate_summary = pd.DataFrame(
    [
        (
            "steam_id",
            len(groups_by_steam_id),
            len(duplicate_steam_groups),
            raw_user_rows - len(groups_by_steam_id),
            dict(sorted(Counter(len(rows) for rows in duplicate_steam_groups.values()).items())),
        ),
        (
            "raw_user_id",
            len(groups_by_raw_user_id),
            len(duplicate_raw_user_groups),
            raw_user_rows - len(groups_by_raw_user_id),
            dict(sorted(Counter(len(rows) for rows in duplicate_raw_user_groups.values()).items())),
        ),
    ],
    columns=["id_column", "unique_values", "duplicate_groups", "duplicate_surplus_rows", "duplicate_group_size_counts"],
)
duplicate_summary

,id_column,unique_values,duplicate_groups,duplicate_surplus_rows,duplicate_group_size_counts
0,steam_id,87625,674,685,"{2: 663, 3: 11}"
1,raw_user_id,87626,673,684,"{2: 662, 3: 11}"


## Second pass: load full rows only for duplicate `steam_id` groups

In [7]:
duplicate_steam_ids = set(duplicate_steam_groups)
duplicate_full_rows: dict[str, list[tuple[int, dict]]] = defaultdict(list)

for line_no, row in iter_user_rows(RAW_USERS_PATH):
    steam_id = norm_id(row.get("steam_id"))
    if steam_id in duplicate_steam_ids:
        duplicate_full_rows[steam_id].append((line_no, row))

loaded_duplicate_rows = sum(len(rows) for rows in duplicate_full_rows.values())
len(duplicate_full_rows), loaded_duplicate_rows

(674, 1359)

## Duplicate case taxonomy

The flags are intentionally not mutually exclusive. The `case` column is a readable primary label.

In [8]:
def classify_duplicate_group(steam_id: str, records: list[tuple[int, dict]]) -> dict:
    summaries = [row_summary(line_no, row) for line_no, row in records]
    raw_user_ids = {summary["raw_user_id"] for summary in summaries}
    user_urls = {summary["user_url"] for summary in summaries}
    reported_counts = {summary["reported_items_count"] for summary in summaries}
    observed_counts = {summary["observed_items_count"] for summary in summaries}
    item_id_digests = {summary["item_id_digest"] for summary in summaries}
    full_digests = {summary["full_snapshot_digest"] for summary in summaries}

    same_raw_user_id = len(raw_user_ids) == 1
    same_user_url = len(user_urls) == 1
    same_reported_items_count = len(reported_counts) == 1
    same_observed_items_count = len(observed_counts) == 1
    same_item_ids = len(item_id_digests) == 1
    same_full_snapshot = len(full_digests) == 1

    if not same_raw_user_id:
        case = "same_steam_id_different_raw_user_id"
    elif same_full_snapshot:
        case = "exact_duplicate_snapshot"
    elif same_item_ids:
        case = "same_items_different_playtime"
    else:
        case = "same_steam_id_different_item_set"

    return {
        "steam_id": steam_id,
        "case": case,
        "rows": len(records),
        "source_lines": [summary["source_line"] for summary in summaries],
        "raw_user_ids": sorted(raw_user_ids),
        "same_raw_user_id": same_raw_user_id,
        "same_user_url": same_user_url,
        "same_reported_items_count": same_reported_items_count,
        "same_observed_items_count": same_observed_items_count,
        "same_item_ids": same_item_ids,
        "same_full_snapshot": same_full_snapshot,
        "min_total_playtime_forever": min(summary["total_playtime_forever"] for summary in summaries),
        "max_total_playtime_forever": max(summary["total_playtime_forever"] for summary in summaries),
        "min_total_playtime_2weeks": min(summary["total_playtime_2weeks"] for summary in summaries),
        "max_total_playtime_2weeks": max(summary["total_playtime_2weeks"] for summary in summaries),
    }


case_df = pd.DataFrame(
    classify_duplicate_group(steam_id, records)
    for steam_id, records in duplicate_full_rows.items()
).sort_values(["case", "steam_id"])

case_counts = case_df["case"].value_counts().rename_axis("case").reset_index(name="groups")
flag_counts = pd.DataFrame(
    [
        ("same_raw_user_id", int(case_df["same_raw_user_id"].sum())),
        ("different_raw_user_id", int((~case_df["same_raw_user_id"]).sum())),
        ("same_user_url", int(case_df["same_user_url"].sum())),
        ("same_reported_items_count", int(case_df["same_reported_items_count"].sum())),
        ("same_observed_items_count", int(case_df["same_observed_items_count"].sum())),
        ("same_item_ids", int(case_df["same_item_ids"].sum())),
        ("same_full_snapshot", int(case_df["same_full_snapshot"].sum())),
        ("different_full_snapshot", int((~case_df["same_full_snapshot"]).sum())),
    ],
    columns=["flag", "groups"],
)

display(case_counts)
display(flag_counts)

,case,groups
0,exact_duplicate_snapshot,657
1,same_items_different_playtime,16
2,same_steam_id_different_raw_user_id,1


,flag,groups
0,same_raw_user_id,673
1,different_raw_user_id,1
2,same_user_url,673
3,same_reported_items_count,674
4,same_observed_items_count,674
5,same_item_ids,674
6,same_full_snapshot,658
7,different_full_snapshot,16


## Example groups

In [9]:
def group_snapshot_table(steam_id: str) -> pd.DataFrame:
    records = duplicate_full_rows[steam_id]
    rows = []
    for line_no, row in records:
        summary = row_summary(line_no, row)
        rows.append(
            {
                "source_line": line_no,
                "steam_id": summary["steam_id"],
                "raw_user_id": summary["raw_user_id"],
                "user_url": summary["user_url"],
                "items_count": summary["observed_items_count"],
                "total_playtime_forever": summary["total_playtime_forever"],
                "total_playtime_2weeks": summary["total_playtime_2weeks"],
                "full_snapshot_digest": summary["full_snapshot_digest"],
            }
        )
    return pd.DataFrame(rows)


def changed_items_table(steam_id: str) -> pd.DataFrame:
    records = duplicate_full_rows[steam_id]
    base_line, base_row = records[0]
    base = item_map(base_row.get("items") or [])
    rows = []

    for line_no, row in records[1:]:
        current = item_map(row.get("items") or [])
        for item_id in sorted(set(base) | set(current)):
            if base.get(item_id) != current.get(item_id):
                before = base.get(item_id)
                after = current.get(item_id)
                rows.append(
                    {
                        "item_id": item_id,
                        "base_line": base_line,
                        "compare_line": line_no,
                        "base_playtime_forever": None if before is None else before[0],
                        "compare_playtime_forever": None if after is None else after[0],
                        "base_playtime_2weeks": None if before is None else before[1],
                        "compare_playtime_2weeks": None if after is None else after[1],
                        "item_name": None if (after or before) is None else (after or before)[2],
                    }
                )
    return pd.DataFrame(rows)


example_ids = {
    "exact_duplicate_snapshot": case_df.loc[case_df["case"].eq("exact_duplicate_snapshot"), "steam_id"].head(1).tolist(),
    "same_items_different_playtime": case_df.loc[case_df["case"].eq("same_items_different_playtime"), "steam_id"].head(1).tolist(),
    "same_steam_id_different_raw_user_id": case_df.loc[case_df["case"].eq("same_steam_id_different_raw_user_id"), "steam_id"].head(1).tolist(),
}

for label, ids in example_ids.items():
    if not ids:
        continue
    steam_id = ids[0]
    print(f"\n{label}: {steam_id}")
    display(group_snapshot_table(steam_id))
    changed = changed_items_table(steam_id)
    if not changed.empty:
        display(changed.head(20))


exact_duplicate_snapshot: 76561197961617681


,source_line,steam_id,raw_user_id,user_url,items_count,total_playtime_forever,total_playtime_2weeks,full_snapshot_digest
0,13048,76561197961617681,multinationalcorporation,http://steamcommunity.com/id/multinationalcorp...,69,264827,1229,37468f22ab1377f460c0b5e1b65319107049aaa5
1,13049,76561197961617681,multinationalcorporation,http://steamcommunity.com/id/multinationalcorp...,69,264827,1229,37468f22ab1377f460c0b5e1b65319107049aaa5



same_items_different_playtime: 76561198011268913


,source_line,steam_id,raw_user_id,user_url,items_count,total_playtime_forever,total_playtime_2weeks,full_snapshot_digest
0,2847,76561198011268913,everybodylovesrayman,http://steamcommunity.com/id/everybodylovesrayman,79,147619,909,559076565c2842ab7ca3aa1ad6d20c2bcfe23e3d
1,31505,76561198011268913,everybodylovesrayman,http://steamcommunity.com/id/everybodylovesrayman,79,147619,881,9ad3237372457473464c8940054535c555606415


,item_id,base_line,compare_line,base_playtime_forever,compare_playtime_forever,base_playtime_2weeks,compare_playtime_2weeks,item_name
0,10190,2847,31505,111,111,111,84,Call of Duty: Modern Warfare 2 - Multiplayer
1,730,2847,31505,4017,4017,315,314,Counter-Strike: Global Offensive



same_steam_id_different_raw_user_id: 76561197994379969


,source_line,steam_id,raw_user_id,user_url,items_count,total_playtime_forever,total_playtime_2weeks,full_snapshot_digest
0,9938,76561197994379969,Panicwinter,http://steamcommunity.com/id/Panicwinter,0,0,0,da39a3ee5e6b4b0d3255bfef95601890afd80709
1,9941,76561197994379969,76561197994379969,http://steamcommunity.com/profiles/76561197994...,0,0,0,da39a3ee5e6b4b0d3255bfef95601890afd80709


## Dedup policy simulation

Candidate policy for canonical Steam users:

1. group raw rows by `steam_id`;
2. choose the snapshot with the largest `total_playtime_forever`;
3. break ties by the earliest `source_line`.

This keeps one internally consistent library snapshot per Steam account.

In [10]:
def choose_snapshot(summaries: list[dict]) -> dict:
    return sorted(summaries, key=lambda row: (-row["total_playtime_forever"], row["source_line"]))[0]


chosen_snapshots = {steam_id: choose_snapshot(rows) for steam_id, rows in groups_by_steam_id.items()}
chosen_source_lines = {row["source_line"] for row in chosen_snapshots.values()}

dedup_summary = pd.DataFrame(
    [
        ("raw_user_rows", raw_user_rows),
        ("canonical_users_after_steam_id_dedup", len(chosen_snapshots)),
        ("removed_duplicate_user_rows", raw_user_rows - len(chosen_snapshots)),
        ("raw_owned_rows", raw_owned_rows),
        ("owned_rows_after_user_snapshot_dedup", sum(row["observed_items_count"] for row in chosen_snapshots.values())),
        ("removed_duplicate_owned_rows", raw_owned_rows - sum(row["observed_items_count"] for row in chosen_snapshots.values())),
    ],
    columns=["metric", "value"],
)
dedup_summary

,metric,value
0,raw_user_rows,88310
1,canonical_users_after_steam_id_dedup,87625
2,removed_duplicate_user_rows,685
3,raw_owned_rows,5153209
4,owned_rows_after_user_snapshot_dedup,5094082
5,removed_duplicate_owned_rows,59127


In [11]:
chosen_duplicate_rows = []
for steam_id, rows in duplicate_steam_groups.items():
    chosen = choose_snapshot(rows)
    chosen_duplicate_rows.append(
        {
            "steam_id": steam_id,
            "raw_duplicate_rows": len(rows),
            "chosen_source_line": chosen["source_line"],
            "chosen_raw_user_id": chosen["raw_user_id"],
            "chosen_total_playtime_forever": chosen["total_playtime_forever"],
            "all_source_lines": [row["source_line"] for row in rows],
            "all_total_playtime_forever": [row["total_playtime_forever"] for row in rows],
        }
    )

chosen_duplicate_df = pd.DataFrame(chosen_duplicate_rows).sort_values(
    ["raw_duplicate_rows", "steam_id"], ascending=[False, True]
)
chosen_duplicate_df.head(20)

,steam_id,raw_duplicate_rows,chosen_source_line,chosen_raw_user_id,chosen_total_playtime_forever,all_source_lines,all_total_playtime_forever
399,76561198027488037,3,14829,76561198027488037,78786,"[14829, 14834, 36238]","[78786, 78786, 78786]"
53,76561198045953692,3,2189,76561198045953692,124178,"[2189, 10666, 28600]","[124178, 124178, 124178]"
131,76561198051777058,3,4374,76561198051777058,0,"[4374, 15835, 38131]","[0, 0, 0]"
2,76561198056857968,3,146,76561198056857968,120129,"[146, 5772, 19474]","[120129, 120129, 120129]"
33,76561198072634646,3,1351,blablabla174,85721,"[1351, 8568, 24866]","[85721, 85721, 85721]"
15,76561198072861800,3,21525,76561198072861800,35196,"[612, 6779, 21525]","[35195, 35195, 35196]"
112,76561198081666970,3,35223,76561198081666970,124786,"[3738, 14297, 35223]","[124766, 124766, 124786]"
403,76561198085989695,3,14833,X03-Suits,128011,"[14833, 14838, 36240]","[128011, 128011, 128011]"
425,76561198094973305,3,15649,76561198094973305,14232,"[15649, 15651, 37761]","[14232, 14232, 14232]"
303,76561198100326818,3,11161,76561198100326818,0,"[11161, 11162, 29501]","[0, 0, 0]"


## Interpretation

- Duplicate `steam_id` rows are repeated snapshots of the same Steam account, not different users.
- Most duplicate groups are exact duplicate library snapshots.
- A smaller set has the same item ids but different playtime values, so choosing one complete snapshot is cleaner than merging item-level playtimes.
- There is a rare alias case where the same `steam_id` appears with different raw `user_id` values; grouping by `steam_id` handles it.
- For the canonical adapter, a reasonable policy is one canonical user per `steam_id`, with one selected source snapshot per user.